# Notebook 3 - Baseline vs graphe

Ce notebook compare :
1. un LLM seul, sans connaissance externe,
2. un prompt enrichi avec des faits récupérés depuis ConceptNet.

On utilise ici un modèle local Ollama, ce qui rend la démonstration simple et reproductible.

In [1]:
import json
import requests

question = "Where should milk be stored?"
choices = ["cupboard", "refrigerator", "desk", "garage"]

baseline_prompt = (
    "You are answering a multiple-choice commonsense question.\n"
    "Select the most plausible answer. Return only the letter.\n\n"
    f"Question: {question}\n"
    "Choices:\n"
    + "\n".join(f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices))
)

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": baseline_prompt,
        "stream": False,
    },
    timeout=120,
)
response.raise_for_status()
print("Réponse baseline :")
print(response.json()["response"])


Réponse baseline :
B


In [2]:
from pathlib import Path
import json
import requests

# Récupération de faits ConceptNet autour des concepts de la question
concepts = ["milk", "store", "refrigerator"]
cache_dir = Path("../data/cache")
cache_dir.mkdir(parents=True, exist_ok=True)

selected_facts = []

for concept in concepts:
    url = f"https://api.conceptnet.io/c/en/{concept}"
    cache_path = cache_dir / f"conceptnet_{concept}.json"
    payload = {"edges": []}

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        payload = response.json()
        cache_path.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
        print(f"ConceptNet live disponible pour '{concept}'.")
    except Exception as exc:
        print(f"ConceptNet live indisponible pour '{concept}' ({exc}).")
        if cache_path.exists():
            payload = json.loads(cache_path.read_text(encoding="utf-8"))
            print(f"Utilisation du cache local pour '{concept}'.")
        else:
            print(f"Aucun cache local disponible pour '{concept}'. Le notebook continue avec une version vide.")
            payload = {"edges": []}

    for edge in payload.get("edges", [])[:3]:
        selected_facts.append(edge)

print("Faits récupérés :")
for fact in selected_facts[:5]:
    print(json.dumps(fact, ensure_ascii=False)[:300])

ConceptNet live indisponible pour 'milk' (502 Server Error: Bad Gateway for url: https://api.conceptnet.io/c/en/milk).
Utilisation du cache local pour 'milk'.
ConceptNet live indisponible pour 'store' (502 Server Error: Bad Gateway for url: https://api.conceptnet.io/c/en/store).
Aucun cache local disponible pour 'store'. Le notebook continue avec une version vide.
ConceptNet live indisponible pour 'refrigerator' (502 Server Error: Bad Gateway for url: https://api.conceptnet.io/c/en/refrigerator).
Aucun cache local disponible pour 'refrigerator'. Le notebook continue avec une version vide.
Faits récupérés :
{"start": {"label": "milk", "language": "en", "@id": "/c/en/milk"}, "end": {"label": "drink", "language": "en", "@id": "/c/en/drink"}, "rel": {"label": "UsedFor"}, "weight": 2.0, "surfaceText": "milk is used for drink"}
{"start": {"label": "milk", "language": "en", "@id": "/c/en/milk"}, "end": {"label": "food", "language": "en", "@id": "/c/en/food"}, "rel": {"label": "IsA"}, "weight"

In [3]:
# Prompt enrichi avec les faits sélectionnés
facts_text = "\n".join(
    f"- {fact.get('start', {}).get('label')} {fact.get('rel', {}).get('label')} {fact.get('end', {}).get('label')}"
    for fact in selected_facts[:5]
)

prompt_graph = (
    "You are answering a multiple-choice commonsense question using relevant knowledge.\n"
    "Use the supplied facts only if they are relevant. Return only the letter.\n\n"
    f"Question: {question}\n"
    "Choices:\n"
    + "\n".join(f"{chr(65+i)}. {choice}" for i, choice in enumerate(choices))
    + "\n\nRelevant knowledge:\n"
    + facts_text
)

response_graph = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": prompt_graph,
        "stream": False,
    },
    timeout=120,
)
response_graph.raise_for_status()
print("Réponse avec graphe :")
print(response_graph.json()["response"])

Réponse avec graphe :
B


### Conclusion

On compare ici le même modèle local sur la même question, mais avec ou sans contexte issu du graphe. C’est le cœur de l’expérience de contribution marginale.

In [4]:
# Vérification de l’état du service Ollama avant l’appel
try:
    health = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama reachable:", health.status_code)
except Exception as exc:
    print("Ollama non disponible localement:", exc)


Ollama reachable: 200


In [5]:
# analyse rapide des résultats texte

def safe_extract_response_text(resp):
    try:
        payload = resp.json()
        return payload.get("response", "")
    except Exception:
        return resp.text.strip() or "[aucune réponse JSON valide]"

baseline_text = safe_extract_response_text(response)
graph_text = safe_extract_response_text(response_graph)

print("Baseline:", baseline_text)
print("Graph:", graph_text)

Baseline: <html>
<head><title>502 Bad Gateway</title></head>
<body bgcolor="white">
<center><h1>502 Bad Gateway</h1></center>
<hr><center>nginx/1.14.0 (Ubuntu)</center>
</body>
</html>
Graph: B


In [7]:
# Exemple de test sur plusieurs questions
#question_bank = [
#    ("Where should milk be stored?", ["cupboard", "refrigerator", "desk", "garage"]),
#    ("Where do people usually keep a toothbrush?", ["bathroom", "kitchen", "garage", "office"]),
#    ("What do people usually use to cut paper?", ["knife", "scissors", "hammer", "book"]),
#    ("Where is a car usually parked?", ["garage", "bedroom", "river", "mountain"]),
#    ("What do you usually wear in winter?", ["coat", "swimsuit", "sandals", "shorts"]),
#    ("Where would you find a stove?", ["kitchen", "library", "beach", "airport"]),
#]

#for question, choices in question_bank:
#    print("\nQuestion:", question)
#    print("Choices:", choices)
